# Daily GMV / Refund / Settlement - exploratory experiments

**Exploratory only - not the production pipeline.** These three daily-aggregate
LinearRegression experiments (GMV, refund, settlement) are the basis for what is
now productionized, but this notebook itself is not re-run by the app:

| Model | Validated here | Production decision | Where |
|---|---|---|---|
| GMV | LinearRegression beat naive (~24% MAE improvement on this CSV) | **Deployed** - `backend/forecaster/gmv.py`, trained from Postgres | `python -m backend.forecaster.train` |
| Refund | LinearRegression ~46% worse than naive | **Rejected** - naive previous-day baseline deployed instead | `backend/forecaster/refund_baseline.py` |
| Settlement | LinearRegression ~55% worse than naive | **Rejected** - naive previous-day baseline deployed instead | `backend/forecaster/settlement_baseline.py` |

Model A (`days_to_settle`) and Model B (`fee_deduction_pct` / `refund_deduction_pct`)
are a separate, per-transaction forecasting pair - see
`model_a_days_to_settle_training.ipynb` and `backend/forecaster/train.py` for those.

See `docs/metrics.md` for the production numbers and `ML/README.md` for how this
notebook relates to the canonical pipeline.


# Daily GMV / Refund / Settlement - exploratory experiments

**Exploratory only - not the production pipeline.** These three daily-aggregate
LinearRegression experiments (GMV, refund, settlement) are the basis for what is
now productionized, but this notebook itself is not re-run by the app:

| Model | Validated here | Production decision | Where |
|---|---|---|---|
| GMV | LinearRegression beat naive (~24% MAE improvement on this CSV) | **Deployed** - , trained from Postgres |  |
| Refund | LinearRegression ~46% worse than naive | **Rejected** - naive previous-day baseline deployed instead |  |
| Settlement | LinearRegression ~55% worse than naive | **Rejected** - naive previous-day baseline deployed instead |  |

Model A () and Model B ( / )
are a separate, per-transaction forecasting pair - see
 and  for those.

See  for the production numbers and  for how this
notebook relates to the canonical pipeline.


### Imports

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import joblib
from pathlib import Path

RANDOM_STATE = 42
TEST_SIZE = 0.2

### data ingestion

In [58]:
df = pd.read_csv('../data/base_transactions.csv')
print(df.shape)
df.head()

(2250, 23)


,txn_id,source,razorpay_payment_id,order_id,txn_amount,payment_method,created_at,day_of_week,is_weekend_or_holiday,merchant_category,...,gst_on_fee_flag,fee_amount,fee_amount_pending,tax_on_fee,fee_pct,days_to_settle,deduction_pct,settled_at,status,true_tax_category
0,TXN00884,synthetic,NaN,order_synth_884,3827.79,card,2025-06-01T00:06:46,6,True,retail,...,True,89.53,False,16.12,0.0234,4,0.0276,2025-06-05T06:06:46,settled,gst_on_fee
1,TXN00106,synthetic,NaN,order_synth_106,10130.91,upi,2025-06-01T02:13:51,6,True,food_delivery,...,True,9.67,False,1.74,0.0010,3,0.0011,2025-06-04T06:13:51,settled,gst_on_fee
2,TXN01922,synthetic,NaN,order_synth_1922,22460.24,upi,2025-06-01T02:47:58,6,True,food_delivery,...,True,114.46,False,20.60,0.0051,3,0.0060,2025-06-04T06:47:58,settled,gst_on_fee
3,TXN01863,synthetic,NaN,order_synth_1863,200.18,card,2025-06-01T04:54:23,6,True,saas,...,True,3.96,False,0.71,0.0198,3,0.0233,2025-06-04T21:54:23,settled,gst_on_fee
4,TXN00439,synthetic,NaN,order_synth_439,23630.03,card,2025-06-01T04:57:25,6,True,saas,...,True,444.31,False,79.98,0.0188,2,0.0222,2025-06-03T08:57:25,settled,gst_on_fee


In [59]:
df.isna().sum()

txn_id                      0
source                      0
razorpay_payment_id      2250
order_id                    0
txn_amount                  0
payment_method              0
created_at                  0
day_of_week                 0
is_weekend_or_holiday       0
merchant_category           0
had_dispute_flag            0
had_refund                  0
refund_amount               0
gst_on_fee_flag             0
fee_amount                  0
fee_amount_pending          0
tax_on_fee                  0
fee_pct                     0
days_to_settle              0
deduction_pct               0
settled_at                  0
status                      0
true_tax_category           0
dtype: int64

In [60]:
df.drop(columns=['razorpay_payment_id'], inplace = True)

In [61]:
df['created_at'] = pd.to_datetime(df['created_at'])

In [62]:
df["date"] = df["created_at"].dt.date
df["date"] = pd.to_datetime(df["date"])

### GMV Model

In [63]:
daily = (
    df.groupby("date")
      .agg(
          total_gmv=("txn_amount", "sum"),
          txn_count=("txn_amount", "count")
      )
      .reset_index()
)

In [64]:
daily

,date,total_gmv,txn_count
0,2025-06-01,367922.80,28
1,2025-06-02,288309.07,22
2,2025-06-03,256403.98,23
3,2025-06-04,408789.08,29
4,2025-06-05,280418.28,24
...,...,...,...
86,2025-08-26,320641.66,23
87,2025-08-27,186217.44,14
88,2025-08-28,325812.19,28
89,2025-08-29,343657.39,25


In [65]:
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["day_of_month"] = daily["date"].dt.day
daily["month"] = daily["date"].dt.month
daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)

In [66]:
daily = pd.get_dummies(
    daily,
    columns=["day_of_week"],
    prefix="dow",
    dtype=int
)

In [67]:
daily["gmv_lag_1"] = daily["total_gmv"].shift(1)

In [68]:
daily["gmv_lag_7"] = daily["total_gmv"].shift(7)

In [69]:
daily.head()

,date,total_gmv,txn_count,day_of_month,month,is_weekend,dow_0,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,gmv_lag_1,gmv_lag_7
0,2025-06-01,367922.80,28,1,6,1,0,0,0,0,0,0,1,NaN,NaN
1,2025-06-02,288309.07,22,2,6,0,1,0,0,0,0,0,0,367922.80,NaN
2,2025-06-03,256403.98,23,3,6,0,0,1,0,0,0,0,0,288309.07,NaN
3,2025-06-04,408789.08,29,4,6,0,0,0,1,0,0,0,0,256403.98,NaN
4,2025-06-05,280418.28,24,5,6,0,0,0,0,1,0,0,0,408789.08,NaN


In [70]:
daily["gmv_rolling_7"] = (
    daily["total_gmv"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily["gmv_rolling_30"] = (
    daily["total_gmv"]
    .shift(1)
    .rolling(30)
    .mean()
)

In [71]:
daily["txn_count_lag_1"] = daily["txn_count"].shift(1)

daily["txn_count_lag_7"] = daily["txn_count"].shift(7)

daily["txn_count_rolling_7"] = (
    daily["txn_count"]
    .shift(1)
    .rolling(7)
    .mean()
)

In [72]:
daily["target_gmv"] = daily["total_gmv"].shift(-1)

In [73]:
daily = daily.dropna()

In [74]:
daily.head()

,date,total_gmv,txn_count,day_of_month,month,is_weekend,dow_0,dow_1,dow_2,dow_3,...,dow_5,dow_6,gmv_lag_1,gmv_lag_7,gmv_rolling_7,gmv_rolling_30,txn_count_lag_1,txn_count_lag_7,txn_count_rolling_7,target_gmv
30,2025-07-01,203505.91,18,1,7,0,0,1,0,0,...,0,0,298198.56,434769.22,315099.071429,314049.502000,27.0,33.0,24.428571,401600.79
31,2025-07-02,401600.79,29,2,7,0,0,0,1,0,...,0,0,203505.91,363247.64,282061.455714,308568.939000,18.0,24.0,22.285714,480077.99
32,2025-07-03,480077.99,35,3,7,0,0,0,0,1,...,0,0,401600.79,245361.14,287540.477143,312345.329667,29.0,18.0,23.000000,212288.98
33,2025-07-04,212288.98,14,4,7,0,0,0,0,0,...,0,0,480077.99,304838.60,321071.455714,319801.130000,35.0,25.0,25.428571,266087.40
34,2025-07-05,266087.40,24,5,7,1,0,0,0,0,...,1,0,212288.98,243036.41,307850.081429,313251.126667,14.0,17.0,23.857143,199525.27


In [75]:
features = [
    "total_gmv",
    "gmv_lag_1",
    "gmv_lag_7",
    "gmv_rolling_7",
    "gmv_rolling_30",
    "txn_count_lag_1",
    "txn_count_lag_7",
    "txn_count_rolling_7",
    "is_weekend"
]

X = daily[features].values
y = daily["target_gmv"].values

In [76]:
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [77]:
from sklearn.linear_model import LinearRegression

gmv_model = LinearRegression()

gmv_model.fit(X_train, y_train)

y_pred = gmv_model.predict(X_test)

In [78]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 87993.94690206916
RMSE: 104470.63385212618


In [79]:
naive_pred = daily["total_gmv"].iloc[split:].values

naive_mae = mean_absolute_error(y_test, naive_pred)

print("Naive MAE:", naive_mae)
print("Linear Regression MAE:", mae)

Naive MAE: 115860.79583333332
Linear Regression MAE: 87993.94690206916


In [80]:
out_dir = Path('artifacts')
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'gmv.joblib'
joblib.dump(gmv_model, out_path)
print(f'Saved {out_path.resolve()}')

Saved /home/sachin/Desktop/Solvent/ML/notebooks/artifacts/gmv.joblib


### Refund Model

In [92]:
refund_daily = (
    df.groupby("date")
      .agg(
          total_refunds=("refund_amount", "sum"),
          refund_count=("had_refund", "sum")
      )
      .reset_index()
)

In [93]:
refund_daily["refund_lag_1"] = (
    refund_daily["total_refunds"].shift(1)
)

refund_daily["refund_lag_7"] = (
    refund_daily["total_refunds"].shift(7)
)

refund_daily["refund_rolling_7"] = (
    refund_daily["total_refunds"]
    .shift(1)
    .rolling(7)
    .mean()
)

refund_daily["refund_rolling_30"] = (
    refund_daily["total_refunds"]
    .shift(1)
    .rolling(30)
    .mean()
)

refund_daily["refund_count_lag_1"] = (
    refund_daily["refund_count"].shift(1)
)

refund_daily["refund_count_lag_7"] = (
    refund_daily["refund_count"].shift(7)
)

refund_daily["refund_count_rolling_7"] = (
    refund_daily["refund_count"]
    .shift(1)
    .rolling(7)
    .mean()
)

In [94]:
refund_daily["target_refund"] = (
    refund_daily["total_refunds"].shift(-1)
)

In [95]:
refund_daily = refund_daily.dropna()

In [96]:
features = [
    "total_refunds",
    "refund_lag_1",
    "refund_lag_7",
    "refund_rolling_7",
    "refund_rolling_30",
    "refund_count_lag_1",
    "refund_count_lag_7",
    "refund_count_rolling_7"
]

X_refund = refund_daily[features].values
y_refund = refund_daily["target_refund"].values

In [97]:
split = int(len(X_refund) * 0.8)

X_train_refund = X_refund[:split]
X_test_refund = X_refund[split:]

y_train_refund = y_refund[:split]
y_test_refund = y_refund[split:]

In [98]:
refund_model = LinearRegression()

refund_model.fit(
    X_train_refund,
    y_train_refund
)

LinearRegression()

In [99]:
y_pred_refund = refund_model.predict(X_test_refund)

In [100]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

refund_mae = mean_absolute_error(
    y_test_refund,
    y_pred_refund
)

refund_rmse = np.sqrt(
    mean_squared_error(
        y_test_refund,
        y_pred_refund
    )
)

print("Refund MAE:", refund_mae)
print("Refund RMSE:", refund_rmse)

Refund MAE: 12536.873799370826
Refund RMSE: 15473.74280287363


In [101]:
naive_refund_pred = refund_daily["total_refunds"].iloc[split:].values

naive_refund_mae = mean_absolute_error(
    y_test_refund,
    naive_refund_pred
)

print("Naive Refund MAE:", naive_refund_mae)
print("Linear Regression Refund MAE:", refund_mae)

improvement = (
    (naive_refund_mae - refund_mae)
    / naive_refund_mae
) * 100

print(f"Improvement: {improvement:.2f}%")

Naive Refund MAE: 8578.121666666666
Linear Regression Refund MAE: 12536.873799370826
Improvement: -46.15%


In [102]:
refund_forecast = refund_daily["total_refunds"].iloc[-1]

In [103]:
print(f"Expected refunds tomorrow: ₹{refund_forecast:,.2f}")

Expected refunds tomorrow: ₹6,929.97


### Settlement Model

In [104]:
df["settled_at"] = pd.to_datetime(df["settled_at"])

In [105]:
df["settlement_date"] = df["settled_at"].dt.date

In [106]:
settlement_daily = (
    df.groupby("settlement_date")
      .agg(
          total_settlement=("txn_amount", "sum"),
          settlement_count=("txn_amount", "count")
      )
      .reset_index()
)

In [107]:
print(df[
    [
        "txn_amount",
        "refund_amount",
        "fee_amount",
        "tax_on_fee",
        "deduction_pct",
        "days_to_settle",
        "settled_at",
        "status"
    ]
].head(10))

   txn_amount  refund_amount  fee_amount  tax_on_fee  deduction_pct  \
0     3827.79            0.0       89.53       16.12         0.0276   
1    10130.91            0.0        9.67        1.74         0.0011   
2    22460.24            0.0      114.46       20.60         0.0060   
3      200.18            0.0        3.96        0.71         0.0233   
4    23630.03            0.0      444.31       79.98         0.0222   
5    15928.74            0.0      292.43       52.64         0.0217   
6      744.08            0.0       12.23        2.20         0.0194   
7     9803.08            0.0        0.00        0.00         0.0000   
8    18662.35            0.0        9.92        1.79         0.0006   
9     7591.89            0.0        0.00        0.00         0.0000   

   days_to_settle          settled_at   status  
0               4 2025-06-05 06:06:46  settled  
1               3 2025-06-04 06:13:51  settled  
2               3 2025-06-04 06:47:58  settled  
3               3 2025

In [108]:
print(df["status"].value_counts())

status
settled     2019
refunded     159
disputed      72
Name: count, dtype: int64


In [109]:
print(df["days_to_settle"].value_counts().sort_index())

days_to_settle
0     178
1     685
2     747
3     365
4     179
5      37
6       5
7      14
8       9
9       7
10      9
11      7
12      3
13      3
14      2
Name: count, dtype: int64


In [110]:
df["net_settlement"] = (
    df["txn_amount"]
    - df["refund_amount"]
    - df["fee_amount"]
    - df["tax_on_fee"]
)

In [111]:
settled_df = df[df["status"] == "settled"].copy()

In [112]:
settled_df["settlement_date"] = (
    pd.to_datetime(settled_df["settled_at"]).dt.date
)

In [113]:
settlement_daily = (
    settled_df.groupby("settlement_date")
    .agg(
        total_settlement=("net_settlement", "sum"),
        settlement_count=("net_settlement", "count")
    )
    .reset_index()
)

In [114]:
settlement_daily["settlement_lag_1"] = (
    settlement_daily["total_settlement"].shift(1)
)

settlement_daily["settlement_lag_7"] = (
    settlement_daily["total_settlement"].shift(7)
)

settlement_daily["settlement_rolling_7"] = (
    settlement_daily["total_settlement"]
    .shift(1)
    .rolling(7)
    .mean()
)

settlement_daily["settlement_rolling_30"] = (
    settlement_daily["total_settlement"]
    .shift(1)
    .rolling(30)
    .mean()
)

settlement_daily["settlement_count_lag_1"] = (
    settlement_daily["settlement_count"].shift(1)
)

settlement_daily["settlement_count_lag_7"] = (
    settlement_daily["settlement_count"].shift(7)
)

settlement_daily["settlement_count_rolling_7"] = (
    settlement_daily["settlement_count"]
    .shift(1)
    .rolling(7)
    .mean()
)

In [115]:
settlement_daily["target_settlement"] = (
    settlement_daily["total_settlement"].shift(-1)
)

In [123]:
settlement_daily = settlement_daily.dropna()

In [124]:
features = [
    "settlement_lag_1",
    "settlement_lag_7",
    "settlement_rolling_7",
    "settlement_rolling_30",
    "settlement_count_lag_1",
    "settlement_count_lag_7",
    "settlement_count_rolling_7"
]

X_settlement = settlement_daily[features].values
y_settlement = settlement_daily["target_settlement"].values

In [125]:
split_settlement = int(len(X_settlement) * 0.8)

X_train_settlement = X_settlement[:split_settlement]
X_test_settlement = X_settlement[split_settlement:]

y_train_settlement = y_settlement[:split_settlement]
y_test_settlement = y_settlement[split_settlement:]

In [126]:
settlement_model = LinearRegression()

settlement_model.fit(
    X_train_settlement,
    y_train_settlement
)

y_pred_settlement = settlement_model.predict(
    X_test_settlement
)

In [127]:
settlement_mae = mean_absolute_error(
    y_test_settlement,
    y_pred_settlement
)

settlement_rmse = np.sqrt(
    mean_squared_error(
        y_test_settlement,
        y_pred_settlement
    )
)

print("Settlement MAE:", settlement_mae)
print("Settlement RMSE:", settlement_rmse)

Settlement MAE: 144930.50230484625
Settlement RMSE: 178439.26624972883


In [128]:
naive_settlement_pred = (
    settlement_daily["total_settlement"]
    .iloc[split_settlement:]
    .values
)

naive_settlement_mae = mean_absolute_error(
    y_test_settlement,
    naive_settlement_pred
)

print("Naive Settlement MAE:", naive_settlement_mae)
print("Linear Regression Settlement MAE:", settlement_mae)

improvement = (
    (naive_settlement_mae - settlement_mae)
    / naive_settlement_mae
) * 100

print(f"Improvement: {improvement:.2f}%")

Naive Settlement MAE: 93583.45307692309
Linear Regression Settlement MAE: 144930.50230484625
Improvement: -54.87%
